In [ ]:
!pip install transformers peft accelerate bitsandbytes datasets -q
!wget -q https://files.pythonhosted.org/packages/18/f3/28d0a3b6c38d766638e1d0b2c0b1adb93c817646842a652a68174861edc5/lcpfn-0.1.3-py3-none-any.whl -O lcpfn-0.1.3-py3-none-any.whl
!pip install --no-deps --ignore-requires-python --force-reinstall ./lcpfn-0.1.3-py3-none-any.whl -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 19.6 MB/s eta 0:00:00


In [1]:
import numpy as np
import torch
import random
import json
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer,
    DataCollatorForLanguageModeling, TrainerCallback,
)
from peft import LoraConfig, get_peft_model

try:
    import lcpfn
    from lcpfn import utils as lcpfn_utils
    LCPFN_AVAILABLE = True
    print("lcpfn imported successfully.")
except ImportError as e:
    print(f"lcpfn not available ({e}) — Arm B will be skipped.")
    LCPFN_AVAILABLE = False

/home/ec2-user/venv_gpu/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


lcpfn imported successfully.


prepare fine tuning

In [2]:
dataset = load_dataset("tatsu-lab/alpaca", split="train")

NUM_RUNS = 10
TRAIN_SIZE = 40
VAL_SIZE = 10

run_configs = []
for run_idx in range(NUM_RUNS):
    random.seed(100 + run_idx)  # different seed per run -> different data -> genuinely distinct curve
    indices = random.sample(range(len(dataset)), TRAIN_SIZE + VAL_SIZE)
    sample = dataset.select(indices)
    train_examples = [
        {"question": row["instruction"] + ((" " + row["input"]) if row["input"] else ""), "answer": row["output"]}
        for row in sample.select(range(TRAIN_SIZE))
    ]
    val_examples = [
        {"question": row["instruction"] + ((" " + row["input"]) if row["input"] else ""), "answer": row["output"]}
        for row in sample.select(range(TRAIN_SIZE, TRAIN_SIZE + VAL_SIZE))
    ]
    run_configs.append({"run_id": f"run_{run_idx}", "train": train_examples, "val": val_examples})

print(f"Prepared {len(run_configs)} independent runs, {TRAIN_SIZE} train / {VAL_SIZE} val examples each.")

Prepared 10 independent runs, 40 train / 10 val examples each.


In [3]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
_tokenizer = None

try:
    import torch_xla.core.xla_model as xm
    _HAS_TPU = True
except ImportError:
    _HAS_TPU = False

_has_cuda = torch.cuda.is_available()
_base_model_dtype = torch.bfloat16 if (_has_cuda or _HAS_TPU) else torch.float32

def _get_tokenizer():
    global _tokenizer
    if _tokenizer is None:
        _tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    return _tokenizer

def _tokenize_examples(examples, tokenizer):
    texts = []
    for ex in examples:
        messages = [
            {"role": "user", "content": ex["question"]},
            {"role": "assistant", "content": ex["answer"]},
        ]
        texts.append(tokenizer.apply_chat_template(messages, tokenize=False))
    return tokenizer(texts, truncation=True, max_length=512, padding="max_length", return_tensors="pt")

def run_real_finetune(train_examples, val_examples, num_epochs=6):
    from datasets import Dataset

    tokenizer = _get_tokenizer()
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, dtype=_base_model_dtype,
        device_map="auto" if _has_cuda else None,
    )
    lora_config = LoraConfig(
        r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    )
    model = get_peft_model(base_model, lora_config)

    def to_hf_dataset(examples):
        tok = _tokenize_examples(examples, tokenizer)
        return Dataset.from_dict({"input_ids": tok["input_ids"], "attention_mask": tok["attention_mask"]})

    train_ds = to_hf_dataset(train_examples)
    val_ds = to_hf_dataset(val_examples)
    collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    val_loss_history = []
    train_loss_history = []

    class HistoryCallback(TrainerCallback):
        def on_log(self, args, state, control, logs=None, **kwargs):
            if logs and "loss" in logs:
                train_loss_history.append(logs["loss"])
        def on_evaluate(self, args, state, control, metrics=None, **kwargs):
            if metrics and "eval_loss" in metrics:
                val_loss_history.append(metrics["eval_loss"])

    args = TrainingArguments(
        output_dir="./smartune-run-checkpoints",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        num_train_epochs=num_epochs,
        learning_rate=2e-4,
        logging_steps=5,
        eval_strategy="epoch",
        save_strategy="no",
        bf16=_has_cuda,
        use_cpu=not (_has_cuda or _HAS_TPU),
        optim="adamw_torch",
        report_to="none",
    )
    trainer = Trainer(
        model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
        data_collator=collator, callbacks=[HistoryCallback()],
    )
    trainer.train()

    del model, base_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return train_loss_history, val_loss_history

In [4]:
import torch; print(torch.__version__); print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0))

2.5.1+cu121
True
NVIDIA A10G


In [5]:
real_curves = {}
for config in run_configs:
    print(f"\n--- Training {config['run_id']} ---")
    train_hist, val_hist = run_real_finetune(config["train"], config["val"], num_epochs=10)
    real_curves[config["run_id"]] = val_hist
    print(f"{config['run_id']} val_loss_history: {val_hist}")

# Save immediately, so a crash later doesn't lose real training results
with open("real_curves.json", "w") as f:
    json.dump(real_curves, f, indent=2)
print("\nSaved real curves to real_curves.json")


--- Training run_0 ---


The model is already on multiple devices. Skipping the move to device specified in `args`.


Epoch,Training Loss,Validation Loss
1,2.180100,1.792081
2,1.709900,1.515820
3,1.573100,1.400412
4,1.380800,1.288242
5,1.131100,1.237549
6,1.133600,1.194401
7,1.049000,1.164419
8,1.047300,1.157782
9,0.924600,1.153304
10,1.144800,1.152106


run_0 val_loss_history: [1.7920806407928467, 1.515819787979126, 1.400411605834961, 1.288241982460022, 1.237549066543579, 1.1944007873535156, 1.164419174194336, 1.157781958580017, 1.1533043384552002, 1.1521055698394775]

--- Training run_1 ---


The model is already on multiple devices. Skipping the move to device specified in `args`.


...

Epoch,Training Loss,Validation Loss
1,1.849800,1.819324
2,1.550300,1.538345
3,1.411500,1.412364
4,1.253900,1.301533
5,1.078200,1.256129
6,0.952600,1.219542
7,1.042100,1.197375
8,1.007000,1.171389
9,0.905300,1.162153
10,1.122300,1.163354


run_8 val_loss_history: [1.8193235397338867, 1.5383445024490356, 1.4123637676239014, 1.301532506942749, 1.2561286687850952, 1.219542145729065, 1.1973745822906494, 1.1713892221450806, 1.1621534824371338, 1.1633541584014893]

--- Training run_9 ---


The model is already on multiple devices. Skipping the move to device specified in `args`.


Epoch,Training Loss,Validation Loss
1,2.143400,2.174061
2,1.685200,1.770046
3,1.564400,1.590838
4,1.249200,1.413832
5,1.111000,1.327437
6,1.141100,1.274894
7,0.937000,1.226668
8,0.973700,1.192051
9,1.025900,1.182672
10,1.011400,1.178779


run_9 val_loss_history: [2.1740610599517822, 1.770045518875122, 1.5908377170562744, 1.413832426071167, 1.3274368047714233, 1.274893879890442, 1.226668119430542, 1.1920514106750488, 1.1826715469360352, 1.1787790060043335]

Saved real curves to real_curves.json


In [6]:
!pip install scipy

In [7]:
# ============================================================
# ARM A
# ============================================================

from scipy.optimize import curve_fit

def _pow3(x, a, b, c):
    return a + b * np.power(x, -c)

def _exp3(x, a, b, c):
    return a + b * np.exp(-c * x)

def _log_power(x, a, b, c):
    return a + (1 - a) / (1 + np.power(np.maximum(x, 1e-6) / max(b, 1e-6), c))

_CURVE_FAMILIES = {
    "pow3": (_pow3, [1.0, 1.0, 0.5]),
    "exp3": (_exp3, [1.0, 1.0, 0.1]),
    "log_power": (_log_power, [0.5, 5.0, 1.0]),
}

def arm_a_predict_curve(val_losses, horizons):
    x_observed = np.arange(1, len(val_losses) + 1, dtype=float)
    y_observed = np.array(val_losses, dtype=float)
    max_horizon = max(horizons)
    x_future = np.arange(len(val_losses) + 1, len(val_losses) + max_horizon + 1, dtype=float)

    fitted = []
    for name, (func, p0) in _CURVE_FAMILIES.items():
        try:
            params, _ = curve_fit(func, x_observed, y_observed, p0=p0, maxfev=5000)
            fitted_y = func(x_observed, *params)
            residuals = fitted_y - y_observed
            sse = float(np.sum(residuals ** 2))
            residual_std = float(np.std(residuals)) + 1e-6
            future_y = func(x_future, *params)
            if np.any(np.isnan(future_y)) or np.any(np.isinf(future_y)):
                continue
            fitted.append((sse, residual_std, future_y))
        except (RuntimeError, ValueError, TypeError):
            continue

    if not fitted:
        return [val_losses[-1]] * len(horizons), [1.0] * len(horizons)

    sses = np.array([f[0] for f in fitted])
    weights = np.exp(-sses / (np.std(sses) + 1e-8))
    weights = weights / weights.sum()
    combined_future = sum(w * f[2] for w, f in zip(weights, fitted))
    combined_std = sum(w * f[1] for w, f in zip(weights, fitted))
    return [combined_future[h - 1] for h in horizons], [combined_std] * len(horizons)

In [8]:
# ============================================================
# ARM B
# ============================================================

_lcpfn_model = None

def _get_lcpfn_model():
    """
    Three separate old-pickle/modern-PyTorch mismatches, all fixed
    defensively inside this function on every call:

    1. torch.load()'s weights_only default changed to True in PyTorch
       2.6+, blocking unpickling of lcpfn's custom TransformerModel class.
    2. Modern nn.TransformerEncoder auto-passes is_causal into layers;
       lcpfn's old custom layer never expected that argument.
    3. Modern nn.GELU expects self.approximate to exist; the checkpoint's
       pickled GELU instances predate that attribute.

    Also force-reloads lcpfn's modules first, so stale unpatched class
    objects from any earlier failed attempt in this same session get
    replaced without needing a full kernel restart.
    """
    global _lcpfn_model
    if _lcpfn_model is not None:
        return _lcpfn_model

    import importlib
    import sys
    import torch.nn as nn

    for mod_name in list(sys.modules):
        if mod_name.startswith("lcpfn"):
            importlib.reload(sys.modules[mod_name])

    import lcpfn as _lcpfn_module
    from lcpfn import utils as _lcpfn_utils_module
    global lcpfn, lcpfn_utils
    lcpfn = _lcpfn_module
    lcpfn_utils = _lcpfn_utils_module

    def _simple_transformer_encoder_forward(self, src, mask=None, src_key_padding_mask=None, is_causal=None):
        output = src
        for mod in self.layers:
            output = mod(output, src_mask=mask, src_key_padding_mask=src_key_padding_mask)
        if self.norm is not None:
            output = self.norm(output)
        return output

    nn.TransformerEncoder.forward = _simple_transformer_encoder_forward
    print("Patched nn.TransformerEncoder.forward (container-level fix applied).")

    nn.GELU.approximate = "none"
    print("Patched nn.GELU class-level default for 'approximate'.")

    original_torch_load = torch.load

    def _load_weights_only_false(*args, **kwargs):
        kwargs["weights_only"] = False
        return original_torch_load(*args, **kwargs)

    torch.load = _load_weights_only_false
    try:
        _lcpfn_model = lcpfn.LCPFN()
        print("LCPFN model constructed successfully.")
    finally:
        torch.load = original_torch_load

    return _lcpfn_model

def _lcpfn_normalizer(val_losses):
    return lcpfn_utils.pfn_normalize(
        lb=torch.tensor(0.0), ub=torch.tensor(float("inf")),
        soft_lb=0.0, soft_ub=torch.tensor(val_losses[0]), minimize=True,
    )

def arm_b_predict_curve(val_losses, horizons):
    if not LCPFN_AVAILABLE:
        raise RuntimeError("lcpfn not installed.")
    model = _get_lcpfn_model()

    x_train = torch.arange(1, len(val_losses) + 1, dtype=torch.float32)
    y_train = torch.tensor(val_losses, dtype=torch.float32)
    x_test = torch.tensor([len(val_losses) + h for h in horizons], dtype=torch.float32)
    normalizer = _lcpfn_normalizer(val_losses)

    y_train_norm = normalizer[0](y_train)
    with torch.no_grad():
        logits = model(x_train=x_train, y_train=y_train_norm, x_test=x_test)
        median = model.model.criterion.icdf(logits, 0.5)
        if median.dim() == 1:
            median = median.unsqueeze(1)  # defensive: ensure 2D regardless of logits' exact shape
        median = normalizer[1](median)

    result = median.squeeze().tolist()
    return result if isinstance(result, list) else [result]

In [9]:
# ============================================================
# ARM C
# ============================================================

def arm_c_fixed_budgets(completed_curves, budgets):
    results = {}
    for budget in budgets:
        achieved = [curve[min(budget, len(curve)) - 1] for curve in completed_curves]
        results[budget] = {
            "mean_final_metric": float(np.mean(achieved)),
            "std_final_metric": float(np.std(achieved)),
            "epochs_used": budget,
        }
    return results

In [10]:
# ============================================================
# Compare
# ============================================================

def mape(actual, pred):
    return 100 * abs(actual - pred) / max(abs(actual), 1e-8)

with open("real_curves.json") as f:
    real_curves = json.load(f)

# Only keep curves long enough to actually test cutoffs against
usable_curves = {name: curve for name, curve in real_curves.items() if len(curve) >= 5}
if len(usable_curves) < len(real_curves):
    print(f"Dropped {len(real_curves) - len(usable_curves)} curve(s) — too short (fewer than 5 eval points).")


cutoffs = [c for c in [3, 4, 5, 6, 7] if c < min(len(v) for v in usable_curves.values())]
horizons = [1, 2, 3]
rows = []
for curve_name, curve in usable_curves.items():
    for cutoff in cutoffs:
        partial = curve[:cutoff]
        valid_horizons = [h for h in horizons if cutoff + h <= len(curve)]
        if not valid_horizons or len(partial) < 2:
            continue
        actuals = [curve[cutoff + h - 1] for h in valid_horizons]

        preds_a, _ = arm_a_predict_curve(partial, valid_horizons)
        for h, pred, actual in zip(valid_horizons, preds_a, actuals):
            rows.append({"curve": curve_name, "cutoff": cutoff, "horizon": h,
                         "arm": "A_domhan", "actual": actual, "pred": pred, "mape": mape(actual, pred)})

        if LCPFN_AVAILABLE:
            try:
                preds_b = arm_b_predict_curve(partial, valid_horizons)
                for h, pred, actual in zip(valid_horizons, preds_b, actuals):
                    rows.append({"curve": curve_name, "cutoff": cutoff, "horizon": h,
                                 "arm": "B_lcpfn", "actual": actual, "pred": pred, "mape": mape(actual, pred)})
            except Exception as e:
                print(f"Arm B failed on {curve_name} at cutoff {cutoff}: {e}")

import pandas as pd
results_df = pd.DataFrame(rows)

print("=" * 60)
print("RESULTS ON REAL FINE-TUNING CURVES")
print("=" * 60)
if len(results_df) > 0:
    print(results_df.groupby("arm")["mape"].agg(["mean", "std", "count"]).to_string(float_format=lambda x: f"{x:.4f}"))
else:
    print("No usable (cutoff, horizon) pairs — runs likely too short. "
          "Increase num_epochs in Cell 4 to get more eval points per curve.")

arm_c_results = arm_c_fixed_budgets(list(usable_curves.values()), budgets=cutoffs if cutoffs else [1])
print("\nArm C (fixed-budget) on real curves:")
for budget, result in arm_c_results.items():
    print(f"  Budget={budget}: mean_final_metric={result['mean_final_metric']:.4f}, std={result['std_final_metric']:.4f}")

# Head-to-head: on how many (curve, cutoff, horizon) cases did each arm actually win?
comparison = results_df.pivot_table(index=["curve", "cutoff", "horizon"], columns="arm", values="mape")
comparison = comparison.dropna()
wins_a = (comparison["A_domhan"] < comparison["B_lcpfn"]).sum()
wins_b = (comparison["B_lcpfn"] < comparison["A_domhan"]).sum()
ties = (comparison["A_domhan"] == comparison["B_lcpfn"]).sum()
print(f"\nHead-to-head: Arm A wins {wins_a}, Arm B wins {wins_b}, ties {ties} (out of {len(comparison)} cases)")


Patched nn.TransformerEncoder.forward (container-level fix applied).
Patched nn.GELU class-level default for 'approximate'.


/tmp/ipykernel_11707/909370707.py:31: OptimizeWarning: Covariance of the parameters could not be estimated
  params, _ = curve_fit(func, x_observed, y_observed, p0=p0, maxfev=5000)


LCPFN model constructed successfully.
RESULTS ON REAL FINE-TUNING CURVES
           mean    std  count
arm                          
A_domhan 2.5122 2.3286    150
B_lcpfn  2.7629 3.0410    150

Arm C (fixed-budget) on real curves:
  Budget=3: mean_final_metric=1.4410, std=0.0984
  Budget=4: mean_final_metric=1.3122, std=0.0878
  Budget=5: mean_final_metric=1.2518, std=0.0818
  Budget=6: mean_final_metric=1.2068, std=0.0813
  Budget=7: mean_final_metric=1.1713, std=0.0798

Head-to-head: Arm A wins 77, Arm B wins 73, ties 0 (out of 150 cases)


In [11]:
# ============================================================
# CELL 7 — Small DA-LCE-style difficulty check (Li & Zhao, 2026, AAAI)
#
# Not a full rebuild — just labels each of your real curves
# Easy/Medium/Hard based on its OWN early dynamics (rate of progress,
# non-linearity, volatility), then checks whether MAPE actually
# correlates with that label the way the DA-LCE paper's own findings
# suggest it should (harder curves -> worse forecasts).
#
# The full, tested version of this logic lives in the main project:
# training/forecasting_comparison.py (compute_difficulty_proxy,
# stratify_by_difficulty, evaluate_stratified_by_difficulty).
# ============================================================

def compute_difficulty_proxy(early_losses):
    y = np.array(early_losses, dtype=float)
    T = len(y)
    if T < 2:
        return {"prog": 0.0, "nonlin": 0.0, "vol": 0.0}
    prog = float((y[-1] - y[0]) / T)
    t = np.arange(T)
    linear_fit = y[0] + (y[-1] - y[0]) / (T - 1) * t
    nonlin = float(np.mean((y - linear_fit) ** 2))
    diffs = np.diff(y)
    vol = float(np.std(diffs)) if len(diffs) > 0 else 0.0
    return {"prog": prog, "nonlin": nonlin, "vol": vol}

def stratify_by_difficulty(curve_names, completed_curves, early_window=3):
    scores = {}
    for name in curve_names:
        proxy = compute_difficulty_proxy(completed_curves[name][:early_window])
        scores[name] = proxy["nonlin"] + proxy["vol"]
    sorted_names = sorted(scores, key=lambda n: scores[n])
    n = len(sorted_names)
    labels = {}
    for i, name in enumerate(sorted_names):
        if i < n / 3:
            labels[name] = "Easy"
        elif i < 2 * n / 3:
            labels[name] = "Medium"
        else:
            labels[name] = "Hard"
    return labels, scores

difficulty_labels, difficulty_scores = stratify_by_difficulty(list(usable_curves.keys()), usable_curves)

print("=" * 60)
print("DIFFICULTY CHECK (DA-LCE-style, curve-derived)")
print("=" * 60)
for name, label in difficulty_labels.items():
    print(f"  {name}: {label}  (nonlin+vol score: {difficulty_scores[name]:.4f})")

if len(results_df) > 0:
    results_df["difficulty"] = results_df["curve"].map(difficulty_labels)
    print("\nMAPE by difficulty label:")
    print(results_df.groupby("difficulty")["mape"].agg(["mean", "count"]).to_string(float_format=lambda x: f"{x:.4f}"))

DIFFICULTY CHECK (DA-LCE-style, curve-derived)
  run_1: Easy  (nonlin+vol score: 0.0750)
  run_2: Easy  (nonlin+vol score: 0.0787)
  run_8: Easy  (nonlin+vol score: 0.0795)
  run_4: Easy  (nonlin+vol score: 0.0812)
  run_0: Medium  (nonlin+vol score: 0.0826)
  run_3: Medium  (nonlin+vol score: 0.1000)
  run_5: Medium  (nonlin+vol score: 0.1070)
  run_7: Hard  (nonlin+vol score: 0.1116)
  run_9: Hard  (nonlin+vol score: 0.1166)
  run_6: Hard  (nonlin+vol score: 0.1193)

MAPE by difficulty label:
             mean  count
difficulty              
Easy       2.1112    120
Hard       3.0986     90
Medium     2.8784     90


In [12]:
# ============================================================
# CELL 8 — Detailed per-example diagnostic
# See exactly which predictions were good/bad, for both arms,
# side by side — needed before trusting any aggregate stat when
# std is this large relative to the mean.
# ============================================================

detail_rows = []
for curve_name, curve in usable_curves.items():
    for cutoff in cutoffs:
        partial = curve[:cutoff]
        valid_horizons = [h for h in horizons if cutoff + h <= len(curve)]
        if not valid_horizons or len(partial) < 2:
            continue
        actuals = [curve[cutoff + h - 1] for h in valid_horizons]

        preds_a, _ = arm_a_predict_curve(partial, valid_horizons)
        try:
            preds_b = arm_b_predict_curve(partial, valid_horizons)
        except Exception:
            preds_b = [None] * len(valid_horizons)

        for h, actual, pa, pb in zip(valid_horizons, actuals, preds_a, preds_b):
            detail_rows.append({
                "curve": curve_name, "cutoff": cutoff, "horizon": h,
                "actual": actual, "pred_a": pa, "pred_b": pb,
                "err_a": abs(pa - actual),
                "err_b": abs(pb - actual) if pb is not None else None,
            })

detail_df = pd.DataFrame(detail_rows)

print("=" * 70)
print("WORST 15 CASES — Arm A")
print("=" * 70)
print(detail_df.nlargest(15, "err_a")[["curve", "cutoff", "horizon", "actual", "pred_a", "err_a"]].to_string(index=False))

print("\n" + "=" * 70)
print("WORST 15 CASES — Arm B")
print("=" * 70)
print(detail_df.nlargest(15, "err_b")[["curve", "cutoff", "horizon", "actual", "pred_b", "err_b"]].to_string(index=False))

print("\n" + "=" * 70)
print("Is it the SAME curves causing trouble for both arms, or different ones?")
print("=" * 70)
worst_a_curves = set(detail_df.nlargest(15, "err_a")["curve"])
worst_b_curves = set(detail_df.nlargest(15, "err_b")["curve"])
print("Curves in both arms' worst-15:", worst_a_curves & worst_b_curves)
print("Only in Arm A's worst-15:", worst_a_curves - worst_b_curves)
print("Only in Arm B's worst-15:", worst_b_curves - worst_a_curves)

/tmp/ipykernel_11707/909370707.py:31: OptimizeWarning: Covariance of the parameters could not be estimated
  params, _ = curve_fit(func, x_observed, y_observed, p0=p0, maxfev=5000)


WORST 15 CASES — Arm A
curve  cutoff  horizon   actual   pred_a    err_a
run_7       3        3 1.145411 1.295325 0.149915
run_6       3        3 1.302024 1.448615 0.146591
run_6       3        2 1.351878 1.469248 0.117370
run_7       3        2 1.200517 1.311484 0.110968
run_9       3        3 1.274894 1.385618 0.110724
run_5       3        3 1.058752 1.158952 0.100200
run_9       3        2 1.327437 1.418427 0.090990
run_6       3        1 1.413642 1.503191 0.089549
run_0       3        3 1.194401 1.280258 0.085857
run_3       3        3 1.278218 1.363962 0.085744
run_4       3        3 1.097424 1.179974 0.082551
run_7       3        1 1.256928 1.338612 0.081683
run_5       3        2 1.107221 1.187206 0.079985
run_4       3        2 1.133383 1.204036 0.070653
run_3       3        2 1.326359 1.394326 0.067967

WORST 15 CASES — Arm B
curve  cutoff  horizon   actual   pred_b    err_b
run_9       3        3 1.274894 1.452441 0.177547
run_9       3        2 1.327437 1.484413 0.156976
run

In [13]:
# ============================================================
# CELL 9 — Export everything needed for documentation
#
# Bundles this run's config, aggregate stats, head-to-head counts,
# Arm C baseline, difficulty breakdown, and worst-case overlap into
# one summary JSON, plus writes/re-confirms the raw CSV and the raw
# per-run val_loss curves — so the whole experiment can be handed
# off for documentation without re-reading printed cell output.
#
# Outputs (all written to the current working directory —
# download them from the Colab/Kaggle file browser after running):
#   - real_curves.json                      (already saved earlier,
#                                             re-confirmed here)
#   - comparison_full_detail.csv            (every curve/cutoff/
#                                             horizon/arm/mape row)
#   - forecasting_experiment_summary.json   (everything below,
#                                             one file, documentation-ready)
# ============================================================

import json as _json
import os as _os

# Re-confirm the CSV export (in case this notebook variant didn't
# already save it after the "Compare" cell).
results_df.to_csv("comparison_full_detail.csv", index=False)

# Epoch count actually trained, read from the data itself rather than
# hardcoded — makes this cell identical across notebook variants that
# used different num_epochs.
_epochs_trained = len(next(iter(usable_curves.values())))

summary = {
    "config": {
        "num_runs": len(usable_curves),
        "epochs_trained_per_run": _epochs_trained,
        "cutoffs_tested": cutoffs,
        "horizons_tested": horizons,
        "model_name": MODEL_NAME,
        "train_size": TRAIN_SIZE,
        "val_size": VAL_SIZE,
        "lcpfn_available": LCPFN_AVAILABLE,
    },
    "aggregate_mape_by_arm": (
        results_df.groupby("arm")["mape"]
        .agg(["mean", "std", "count"])
        .round(4)
        .to_dict(orient="index")
    ),
    "mape_by_cutoff_and_arm": (
        results_df.groupby(["cutoff", "arm"])["mape"]
        .agg(["mean", "std", "count"])
        .round(4)
        .reset_index()
        .to_dict(orient="records")
    ),
    "head_to_head": {
        "arm_a_wins": int(wins_a),
        "arm_b_wins": int(wins_b),
        "ties": int(ties),
        "total_cases": int(len(comparison)),
    },
    "arm_c_fixed_budget": {
        str(budget): {
            "mean_final_metric": round(result["mean_final_metric"], 4),
            "std_final_metric": round(result["std_final_metric"], 4),
        }
        for budget, result in arm_c_results.items()
    },
    "difficulty_labels": difficulty_labels,
    "difficulty_scores": {k: round(v, 4) for k, v in difficulty_scores.items()},
    "mape_by_difficulty": (
        results_df.assign(difficulty=results_df["curve"].map(difficulty_labels))
        .groupby("difficulty")["mape"]
        .agg(["mean", "count"])
        .round(4)
        .to_dict(orient="index")
    ),
    "worst_case_overlap": {
        "in_both_arms_worst_15": sorted(worst_a_curves & worst_b_curves),
        "only_in_arm_a_worst_15": sorted(worst_a_curves - worst_b_curves),
        "only_in_arm_b_worst_15": sorted(worst_b_curves - worst_a_curves),
    },
}

with open("forecasting_experiment_summary.json", "w") as f:
    _json.dump(summary, f, indent=2)

print("Export complete. Files in the current directory:")
for fname in ["real_curves.json", "comparison_full_detail.csv", "forecasting_experiment_summary.json"]:
    exists = _os.path.exists(fname)
    size = _os.path.getsize(fname) if exists else 0
    print(f"  {'OK' if exists else 'MISSING'}  {fname}  ({size:,} bytes)")

print("\nDownload all three from the file browser (left sidebar in Colab/Kaggle),")
print("or right-click each file individually.")
print("\nSummary preview:")
print(_json.dumps(summary["config"], indent=2))
print(_json.dumps(summary["head_to_head"], indent=2))


Export complete. Files in the current directory:
  OK  real_curves.json  (2,545 bytes)
  OK  comparison_full_detail.csv  (24,128 bytes)
  OK  forecasting_experiment_summary.json  (3,274 bytes)

Download all three from the file browser (left sidebar in Colab/Kaggle),
or right-click each file individually.

Summary preview:
{
  "num_runs": 10,
  "epochs_trained_per_run": 10,
  "cutoffs_tested": [
    3,
    4,
    5,
    6,
    7
  ],
  "horizons_tested": [
    1,
    2,
    3
  ],
  "model_name": "Qwen/Qwen2.5-1.5B-Instruct",
  "train_size": 40,
  "val_size": 10,
  "lcpfn_available": true
}
{
  "arm_a_wins": 77,
  "arm_b_wins": 73,
  "ties": 0,
  "total_cases": 150
}
